# 0. Sample

In [1]:
from ase.visualize import view

from src.sample import sample

In [2]:
help(sample)

Help on function sample in module src.sample:

sample(num_samples: int = 10000, batch_size: int = 2000, compositions: list | None = None, text_prompts: list | None = None, num_atom_distribution: str = 'mp-20', ldm_ckpt_path: str | pathlib.Path | None = None, vae_ckpt_path: str | pathlib.Path | None = None, output_dir: str = 'outputs', sampler: str = 'ddim', sampling_steps: int = 50, cfg_scale: float = 2.0, device: str | None = None, save_json: bool = True)
    Sample crystal structures using a pre-trained LDM model.
    
    if compositions are provided, it performs the CSP (Crystal Structure Prediction) task.
    elif text_prompts are provided, it performs the TSP (Text-to-Structure Prediction) task.
    If neither compositions nor text_prompts are provided, it performs the DNG (De Novo Generation) task.
    
    :param num_samples: Total number of samples to generate, defaults to 10000
    :param batch_size: Number of samples to generate in each batch, defaults to 2000
    :param com

In [3]:
# Sample with Default model trained with alex-mp-20
gen_atoms_list = sample(
    num_samples=1000,
    batch_size=500,
    output_dir="outputs/alex-mp-20",
)

Using device: cuda
Loaded VAE from checkpoints/v0.0.1/alex_mp_20/vae/dng_j1jgz9t0_v1.ckpt
Loaded model from checkpoints/v0.0.1/alex_mp_20/ldm/ldm_rl_dng_tuor5vgd.ckpt
DNG task: 1000 samples
The sampled cif files will be saved in directory: 'outputs/alex-mp-20'
Generating batch #1 with 500 samples.
Using ddim sampler_fn with 50 timesteps.


/home/cao/opensource/cailiao/chemeleon2/.venv/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:210: Attribute 'encoder' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['encoder'])`.
/home/cao/opensource/cailiao/chemeleon2/.venv/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:210: Attribute 'decoder' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['decoder'])`.
/home/cao/opensource/cailiao/chemeleon2/.venv/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:210: Attribute 'denoiser' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['denoiser'])`.
/home/cao/opensource/cailiao/chemeleon2/src/sample.py:127: UserWarning: Output directory 'outputs/

  0%|          | 0/50 [00:00<?, ?it/s]

Generating batch #2 with 500 samples.
Using ddim sampler_fn with 50 timesteps.


  0%|          | 0/50 [00:00<?, ?it/s]

/home/cao/opensource/cailiao/chemeleon2/.venv/lib/python3.11/site-packages/pymatgen/core/structure.py:3107: UserWarning: Issues encountered while parsing CIF: 1 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = parser.parse_structures(primitive=primitive)[0]
/home/cao/opensource/cailiao/chemeleon2/.venv/lib/python3.11/site-packages/pymatgen/core/structure.py:3107: UserWarning: Issues encountered while parsing CIF: 2 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = parser.parse_structures(primitive=primitive)[0]


The 2000 generated structures saved in JSON format at: outputs/alex-mp-20/generated_structures.json.gz


In [4]:
view(gen_atoms_list[0], viewer="ngl")

In [5]:
# Sample with mp-20 model
from src.utils.checkpoint import get_checkpoint

vae_ckpt_path = get_checkpoint("mp_20_vae")
ldm_ckpt_path = get_checkpoint("mp_20_ldm")

gen_structures_mp20 = sample(
    num_samples=1000,
    batch_size=500,
    ldm_ckpt_path=ldm_ckpt_path,
    vae_ckpt_path=vae_ckpt_path,
    output_dir="outputs/mp-20",
)

KeyError: 'mp_20_ldm'

# 1. Evaluate

> **Note**: Before running metrics for evaluation, see [Prerequisites in Evaluation Guide](docs/EVALUATION.md#prerequisites) for setup instructions (e.g., downloading reference datasets, pre-computed features).

In [ ]:
from monty.serialization import loadfn

from src.utils.metrics import Metrics

In [ ]:
help(Metrics)

In [ ]:
# Load generated structures from [0.Sample] step
gen_structures = loadfn("outputs/alex-mp-20/generated_structures.json.gz")

# Initialize metrics calculator
metrics = Metrics(
    metrics=["unique", "novel", "e_above_hull"],
    reference_dataset="mp-20",
    phase_diagram="mp-all",
    metastable_threshold=0.1,
)

FileNotFoundError: [Errno 2] No such file or directory: '/home/cao/opensource/cailiao/chemeleon2/benchmarks/assets/mp_20_all_structure.json.gz'

In [ ]:
results = metrics.compute(gen_structures)

In [ ]:
# Calculate mSUN score (percentage that are unique, novel, AND metastable)
msun_score = (
    results["unique"] & results["novel"] & results["is_metastable"]
).mean() * 100
print(f"mSUN Score: {msun_score:.2f}%")
print(f"Uniqueness: {results['unique'].mean():.2%}")
print(f"Novelty: {results['novel'].mean():.2%}")
print(f"Metastable: {results['is_metastable'].mean():.2%}")

# Convert to DataFrame for analysis
df = metrics.to_dataframe()
df.head()

# Save results to CSV
df.to_csv("outputs/alex-mp-20/metrics_results.csv")